# Monthly vs yearly

This notebook compares **hindfoot length** and **weight** when the data are grouped by month versus by year.

The visual goal is to see how much detail is visible in monthly averages compared to the smoother yearly averages. The notebook first studies the full dataset, then repeats the same view for the **Dipodomys** genus because it has many observations.

## Load the data

This cell reads the merged dataset into pandas.

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv("../../output/merged_full.csv")
df.head()

## Prepare a simple dataset

This cell keeps only the time columns, taxonomy columns, and the two measurement columns we are analyzing.

In [ ]:
time_df = df[['year', 'month', 'genus', 'species_name', 'hindfoot_length', 'weight']].copy()

for col in ['genus', 'species_name']:
    time_df[col] = time_df[col].astype('string').str.strip()

time_df = time_df.dropna(subset=['year', 'month', 'genus', 'species_name']).copy()
time_df = time_df.loc[time_df['month'].between(1, 12)& time_df['genus'].ne('')& time_df['species_name'].ne('')].copy()

time_df['year'] = time_df['year'].astype(int)
time_df['month'] = time_df['month'].astype(int)
time_df

## Build Monthly Summaries

This cell calculates monthly mean hindfoot length and monthly mean weight for the entire dataset. Each row represents one month in one year.

In [ ]:
monthly_df = (
    time_df.groupby(['year', 'month'], as_index=False)
    .agg(
        mean_hindfoot_length=('hindfoot_length', 'mean'),
        mean_weight=('weight', 'mean'),
        n_observations=('weight', 'size'),
    )
)

monthly_df['date'] = pd.to_datetime(
    dict(year=monthly_df['year'], month=monthly_df['month'], day=1)
)

monthly_df

## Build Yearly Summaries

This cell calculates yearly mean hindfoot length and yearly mean weight for the entire dataset. These yearly averages will be overlaid as line plots on top of the monthly scatter plots.

In [ ]:
yearly_df = (
    time_df.groupby(['year'], as_index=False)
    .agg(
        mean_hindfoot_length=('hindfoot_length', 'mean'),
        mean_weight=('weight', 'mean'),
        n_observations=('weight', 'size'),
    )
)

yearly_df['date'] = pd.to_datetime(
    dict(year=yearly_df['year'], month=7, day=1)
)

yearly_df.head(10)

## Monthly vs Yearly Visualizations

The plots below compare monthly aggregation against yearly aggregation for the full dataset. Monthly averages are shown as scatter points, while yearly averages are shown as an overlaid line. This keeps the plot focused on the difference between detailed monthly movement and smoother yearly trends.

In [ ]:

# FIG 2.1
display(Image(filename="../../images/fig_2_1.png"))


The full-dataset plot shows the overall relationship between month-level and year-level aggregation. The monthly points reveal short-term variation, while the yearly line shows the broader trend after those month-to-month changes are smoothed together.

## Dipodomys Monthly vs Yearly View

The same visualization is repeated for **Dipodomys** only. This genus has many observations, so it is a useful focused example for comparing monthly and yearly aggregation without splitting the chart into every genus.

In [ ]:

# FIG 2.2
display(Image(filename="../../images/fig_2_2.png"))


In the Dipodomys plot, the yearly line gives a cleaner long-term trend, while the monthly points show the spread around that trend. This makes it easier to decide whether yearly aggregation is detailed enough for the question being asked, or whether the monthly values tell a different story.

## Dipodomys Weight by Month Within Each Year

The next plot focuses only on **mean weight** for Dipodomys. Hindfoot length is not included here because it is a body-size measurement that should not change as much within a single calendar year. Weight is more likely to respond to short-term conditions, so it is the better measurement for checking whether monthly patterns repeat across years or look more random.

In [ ]:

# FIG 2.3
display(Image(filename="../../images/fig_2_3.png"))


This plot treats each year as its own month-by-month line. If many yearly lines rise and fall in similar months, that suggests a repeated seasonal pattern. If the gray yearly lines move in different directions from year to year, then the monthly changes may be more random or driven by year-specific conditions. The black line gives the average monthly pattern across all years, making it easier to see whether there is a repeated tendency underneath the yearly noise.

## Conclusion

- Monthly aggregation is better for seeing short-term variation and possible seasonal movement.
- Yearly aggregation is better for seeing the broader trend because it smooths over month-to-month changes.
- Looking at the full dataset gives the overall pattern, while the Dipodomys-only view gives a cleaner focused example with many data points.
- The Dipodomys month-within-year weight plot adds another layer by showing whether monthly weight trends repeat across years or change randomly from year to year.
- Hindfoot length is not used for that year-by-month seasonal plot because it is less likely to change meaningfully within one calendar year; weight is more useful for studying short-term monthly patterns.
- If the monthly scatter points stay close to the yearly line, yearly aggregation is probably enough. If the points spread widely around the line, or if the within-year lines show repeated seasonal movement, the monthly aggregation may be more useful.